
# 練習問題: GPUでベクトル加算 (map(to:) と map(from:))

## 目標

GPUで配列計算をするとき, **どのデータをGPUへ送り (to), どの結果をGPUから戻すか (from)** を `map` 節で正しく指定できるようになる.

## 課題

`vecadd.cpp` (または `vecadd.f90`) は, 配列 `a`, `b` を初期化し, `c[i] = a[i] + b[i]` を計算して検算する.
計算本体 (オフロードするループ) が抜けているので, 現状では `c` が初期値 `-1` のままで検算に失敗する.

`TODO` の箇所に **ループをGPUにオフロードして `c[i] = a[i] + b[i]` を計算する処理** を書け. その際, 入力 `a`, `b` はGPUへ送り, 結果 `c` はGPUから受け取るよう `map` 節を指定する.

- C++: `#pragma omp target teams distribute parallel for map(to: a[0:n], b[0:n]) map(from: c[0:n])` とその直後の `for` ループ.
- Fortran: `!$omp target teams distribute parallel do map(to: a, b) map(from: c)` … `do` ループ … `!$omp end target teams distribute parallel do`.

考えどころ: `a`, `b` は入力なので送るだけ (`to`), `c` は結果なので戻すだけ (`from`). `map(tofrom:)` でも動くが, 余計な転送を避けるには `to`/`from` を使い分けるのがよい.

## コンパイルと実行

```
# C++
nvc++ -mp=gpu vecadd.cpp -o vecadd.exe

# Fortran
nvfortran -mp=gpu vecadd.f90 -o vecadd.exe
```

GPUは計算ノードにのみ搭載されているので, `%%bash_submit` でジョブとして投入する.

```
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:10:00

./vecadd.exe 1000
```

## 期待される結果

正しくオフロード・転送できていれば検算に通り `OK` が表示される.

```
OK: c[0] = 0, c[999] = 2997
```

- オフロードを書く前は `c` が `-1` のままで `NG` になる.
- `map(from: c)` を忘れると, GPU上では計算できていてもホスト側の `c` が更新されず `NG` になることも確かめてみよ.



# 1. ツールの読み込み
- AIチュータ及びジョブ投入ツールの読み込み (カーネル起動後に一度実行すればよい)
  - `heytutor` : `%%hey` でAIチュータに質問できるようになる (使い方は末尾を参照)
  - `wisteria_submit` : `%%bash_submit` (先頭に `#PJM ...` を書く) でジョブ投入できるようになる


In [ ]:
import heytutor
import wisteria_submit

# 2. C++ ベースコード

In [ ]:
%%writefile_ vecadd.cpp
#include <cstdio>
#include <cstdlib>

int main(int argc, char ** argv) {
  long n = (argc > 1 ? atol(argv[1]) : 1000);
  double * a = (double *)malloc(sizeof(double) * n);
  double * b = (double *)malloc(sizeof(double) * n);
  double * c = (double *)malloc(sizeof(double) * n);
  for (long i = 0; i < n; i++) { a[i] = i; b[i] = 2 * i; c[i] = -1.0; }

  /* c[i] = a[i] + b[i] を GPU で計算する */
  // TODO: ループをGPUにオフロードして c[i]=a[i]+b[i] を計算せよ. a,b は map(to:), 結果 c は map(from:) で受け取る.

  /* 検算 */
  long err = 0;
  for (long i = 0; i < n; i++) {
    if (c[i] != a[i] + b[i]) err++;
  }
  if (err == 0) {
    printf("OK: c[0] = %.0f, c[%ld] = %.0f\n", c[0], n - 1, c[n - 1]);
  } else {
    printf("NG: %ld 要素が不正 (例: c[0] = %.0f, 正解は %.0f)\n", err, c[0], a[0] + b[0]);
  }
  free(a); free(b); free(c);
  return 0;
}

## 2-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvc++ -fast -mp=gpu vecadd.cpp -o vecadd_cpp.exe

## 2-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./vecadd_cpp.exe

# 3. Fortran ベースコード

In [ ]:
%%writefile_ vecadd.f90
program vecadd
  character(len=32) :: arg
  integer(8) :: n, i, err
  real(8), allocatable :: a(:), b(:), c(:)
  n = 1000
  if (command_argument_count() >= 1) then
     call get_command_argument(1, arg); read (arg, *) n
  end if
  allocate(a(n), b(n), c(n))
  do i = 1, n
     a(i) = i; b(i) = 2 * i; c(i) = -1.0d0
  end do

  ! c(i) = a(i) + b(i) を GPU で計算する
  ! TODO: ループをGPUにオフロードして c(i)=a(i)+b(i) を計算せよ. a,b は map(to:), 結果 c は map(from:) で受け取る.

  ! 検算
  err = 0
  do i = 1, n
     if (c(i) /= a(i) + b(i)) err = err + 1
  end do
  if (err == 0) then
     print "(a,f0.0,a,i0,a,f0.0)", "OK: c(1) = ", c(1), ", c(", n, ") = ", c(n)
  else
     print "(a,i0,a)", "NG: ", err, " 要素が不正"
  end if
end program vecadd

## 3-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvfortran -fast -mp=gpu vecadd.f90 -o vecadd_f90.exe

## 3-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./vecadd_f90.exe


# 4. AIチュータへの質問の仕方 (参考)
- 先頭で `import heytutor` 済みなら, セルに `%%hey` と書いて質問できる。
- ChatGPTなどと同様に自由に質問してよい。ただし「答えをそのまま教えて」などは自制すること。
- セル内で使える変数 (自動で展開される):
  - `{file:FILENAME}` : _FILENAME_ の中身 (例: `{file:problem.md}`, `{file:vecadd.cpp}`)
  - `{bash[-1]}` : 最後に実行した `%%bash_` セルの入力・出力, `{bash[-2]}` = その前, ...
- 以下は質問例 (必要に応じてコピーして使う。Fortranなら `.cpp` を `.f90` に書き換える)。

## 4-1. 一般的な質問

In [ ]:
%%hey

C++の関数定義の文法どんなだっけ?

## 4-2. この問題に関するヒント

In [ ]:
%%hey

この問題に関するヒントを教えて

問題:
{file:problem.md}

## 4-3. 困ったときのヘルプ
- コンパイル時や実行時のエラー直後に実行するとエラーに関するヘルプが得られる。

In [ ]:
%%hey

以下のエラーが出た。何が間違い?

プログラム:
{file:vecadd.cpp}

コマンドと実行結果:
{bash[-1]}

## 4-4. フィードバック
- 答えが出た後も, 無駄なところやより良いやり方がないかを聞くことを推奨。

In [ ]:
%%hey

私の答に対するフィードバックをください。

問題:
{file:problem.md}

私の答:
{file:vecadd.cpp}